# Advanced Retrieval-Augmented Generation: Techniques and Architectures
## Industry Case Study — NimbusCloud Enterprise Support Knowledge Base

**Business context.** NimbusCloud is a fictional enterprise SaaS company offering cloud
storage, file collaboration, and API-based integrations to business customers. The support
and solutions engineering teams maintain a knowledge base covering billing, security,
compliance, API usage, troubleshooting, and onboarding. Customer support agents and an
internal support chatbot need to answer natural-language questions against this knowledge
base accurately, with correct citations, and with graceful handling of ambiguous or
multi-part questions.

**Objective of this notebook.** Build a Retrieval-Augmented Generation (RAG) system from
first principles — without any orchestration framework (no LangChain, no agent framework)
— and progressively layer in the retrieval, query, context, and architectural improvements
that a production RAG system in industry would use. Each technique is implemented directly
with the OpenAI Python SDK, `rank_bm25`, `sentence-transformers`, and `numpy`, and is
demonstrated on the same knowledge base so the effect of each technique is directly
comparable.

**Models used**
- Embedding model: `text-embedding-3-small`
- Generation / reasoning model: `gpt-4o-mini`

**Credentials.** The OpenAI API key is read from Colab Secrets (`google.colab.userdata`),
never hard-coded.

**Notebook structure**

| Section | Techniques |
|---|---|
| 0 | Environment setup, case-study knowledge base, core utilities |
| 1 | Baseline naive RAG (control group) |
| 2 | Retrieval improvements: Dense vs Sparse, Hybrid Search, Reciprocal Rank Fusion, Re-ranking, Cross-Encoder Reranker |
| 3 | Query improvements: Query Rewriting, Query Expansion, Multi-Query Retrieval, HyDE, Query Decomposition, Step-back Prompting |
| 4 | Context improvements: Contextual Compression, Chunk Reordering, Parent-Child Retrieval, Metadata Filtering |
| 5 | Advanced architectures: Multi-stage Retrieval, Self-RAG, Corrective RAG (CRAG) |
| 6 | Production pipeline: putting it all together + comparative evaluation |


**Note on standalone execution.** This notebook is one part of a series covering advanced RAG techniques for the NimbusCloud case study. It repeats the setup and any helper functions it depends on from earlier notebooks in the series so that it can be run top-to-bottom on its own, without needing to run the other notebooks first.

---
# Section 0 — Environment Setup

Install dependencies and configure API access. No orchestration framework (LangChain) and
no agent framework is used anywhere in this notebook; every retrieval and generation step is
implemented directly against the OpenAI API and standard retrieval libraries.


In [ ]:
!pip install -q openai rank_bm25 sentence-transformers numpy pandas scikit-learn tiktoken


In [ ]:
import os
import json
import re
import time
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

from google.colab import userdata
from openai import OpenAI

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

pd.set_option("display.max_colwidth", 120)
print("Client configured. Embedding model:", EMBED_MODEL, "| Chat model:", CHAT_MODEL)


## Core LLM and embedding utilities

These two wrapper functions are the only interface to the OpenAI API used throughout the
notebook. Every technique below is composed from `embed_texts` and `chat`.


In [ ]:
def embed_texts(texts: List[str]) -> np.ndarray:
    '''Embed a list of texts with text-embedding-3-small. Returns an (N, D) float32 array,
    L2-normalized so that dot product equals cosine similarity.'''
    if len(texts) == 0:
        return np.zeros((0, 1536), dtype=np.float32)
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1e-8
    return vecs / norms


def chat(prompt: str, system: Optional[str] = None, temperature: float = 0.0,
         model: str = CHAT_MODEL, max_tokens: int = 600) -> str:
    '''Single-turn chat completion wrapper used for every LLM-driven step
    (rewriting, expansion, reranking, grading, compression, generation, etc.).'''
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()


## Case-study knowledge base

The corpus below models a realistic enterprise support knowledge base: 24 parent documents
across six categories (Billing, Security & Compliance, API & Integrations, Troubleshooting,
Onboarding, Data Governance), each tagged with metadata (`category`, `product_area`,
`updated`) so that metadata filtering later in the notebook has something real to filter on.

Each parent document is also split into smaller child chunks. Dense/sparse/hybrid retrieval
in Sections 2-5 operates on the **child chunks**; Section 4's Parent-Child Retrieval
technique demonstrates retrieving on children but returning the parent for generation.


In [ ]:
PARENT_DOCS = [
    {"doc_id": "P01", "category": "Billing", "product_area": "Subscriptions", "updated": "2025-11-02",
     "title": "Monthly vs Annual Billing Cycles",
     "text": "NimbusCloud offers monthly and annual billing cycles for all paid tiers. Annual "
             "billing is invoiced once per year and carries a 15 percent discount compared to "
             "paying monthly. Customers can switch from monthly to annual billing at any time "
             "from the Billing tab in the admin console; the switch takes effect at the start of "
             "the next billing period and is prorated. Switching from annual back to monthly is "
             "only permitted at the end of the current annual term to avoid partial refunds."},
    {"doc_id": "P02", "category": "Billing", "product_area": "Invoicing", "updated": "2025-09-14",
     "title": "Invoice Delivery and Payment Methods",
     "text": "Invoices are generated on the first day of each billing cycle and emailed to the "
             "account's designated billing contact as a PDF. NimbusCloud accepts payment by credit "
             "card, ACH bank transfer, and wire transfer for annual contracts above 10,000 dollars. "
             "Failed credit card payments trigger three automatic retries over seven days before "
             "the account is moved to a past-due state, which restricts write access but preserves "
             "read access to existing files for 30 days."},
    {"doc_id": "P03", "category": "Billing", "product_area": "Refunds", "updated": "2025-08-01",
     "title": "Refund and Proration Policy",
     "text": "Refunds for annual plans are available on a prorated basis within the first 30 days "
             "of the annual term if the customer downgrades or cancels. After the first 30 days, "
             "annual plans are non-refundable but can be cancelled with the remaining term "
             "honored through its expiration date. Monthly plans are billed in arrears for usage "
             "and are not eligible for refunds since the charge reflects services already consumed."},
    {"doc_id": "P04", "category": "Billing", "product_area": "Overages", "updated": "2025-10-20",
     "title": "Storage Overage Charges", 
     "text": "Each paid tier includes a storage allotment; usage beyond the allotment is billed as "
             "an overage at 0.03 dollars per gigabyte per month, calculated on the average daily "
             "usage across the billing period. Overage charges appear as a separate line item on "
             "the next invoice. Admins can set a soft usage alert at 80 percent of allotment and a "
             "hard cap that blocks new uploads once the allotment is exceeded, configurable in "
             "Account Settings under Storage Limits."},

    {"doc_id": "P05", "category": "Security", "product_area": "Encryption", "updated": "2025-12-01",
     "title": "Data Encryption at Rest and in Transit",
     "text": "All files stored in NimbusCloud are encrypted at rest using AES-256, with encryption "
             "keys managed through a dedicated key management service and rotated every 90 days. "
             "Data in transit between client applications and NimbusCloud servers is protected "
             "with TLS 1.2 or higher. Enterprise customers on the Scale and Enterprise tiers can "
             "optionally enable customer-managed encryption keys (CMEK), giving them direct control "
             "over key rotation and revocation."},
    {"doc_id": "P06", "category": "Security", "product_area": "Authentication", "updated": "2025-11-18",
     "title": "Single Sign-On (SSO) Configuration",
     "text": "SSO is available on the Business and Enterprise tiers and supports SAML 2.0 and "
             "OpenID Connect. To configure SSO, an admin uploads the identity provider metadata "
             "XML in Security Settings, maps user attributes to NimbusCloud roles, and enables "
             "enforced SSO to require all members to authenticate through the identity provider. "
             "Just-in-time provisioning can automatically create NimbusCloud accounts for new "
             "identity provider users on first login."},
    {"doc_id": "P07", "category": "Security", "product_area": "Compliance", "updated": "2025-07-22",
     "title": "GDPR Data Subject Requests",
     "text": "NimbusCloud acts as a data processor under GDPR for content that customers store on "
             "the platform. Data subject access, correction, and deletion requests should be "
             "submitted by the customer's data controller through the Compliance Center. "
             "NimbusCloud responds to verified deletion requests within 30 days, permanently "
             "removing the data from primary storage within 7 days and from backups within the "
             "next backup rotation cycle, which completes within 90 days."},
    {"doc_id": "P08", "category": "Security", "product_area": "Incident Response", "updated": "2025-10-05",
     "title": "Security Incident Notification Policy",
     "text": "In the event of a confirmed security incident affecting customer data, NimbusCloud "
             "notifies affected account administrators within 72 hours of confirmation, consistent "
             "with contractual and regulatory notification windows. Notifications include the "
             "nature of the incident, data categories affected, remediation steps taken, and "
             "recommended actions for the customer. Customers on Enterprise tier are also assigned "
             "a dedicated incident liaison for the duration of the response."},
    {"doc_id": "P09", "category": "Security", "product_area": "Access Control", "updated": "2025-09-30",
     "title": "Role-Based Access Control (RBAC)",
     "text": "NimbusCloud supports four built-in roles: Viewer, Editor, Admin, and Owner. Roles can "
             "be assigned at the workspace level or overridden at the folder level for more granular "
             "control. Custom roles with fine-grained permissions such as 'can share externally' or "
             "'can view audit logs' are available on the Enterprise tier. Role changes are recorded "
             "in the audit log with the acting admin, timestamp, and previous role."},

    {"doc_id": "P10", "category": "API", "product_area": "Authentication", "updated": "2025-11-25",
     "title": "API Authentication and Token Management",
     "text": "The NimbusCloud REST API authenticates requests using bearer tokens generated from "
             "the Developer Console. Tokens can be scoped to read-only or read-write access and to "
             "specific workspaces. Tokens do not expire automatically but can be revoked at any "
             "time; revoked tokens are rejected within 60 seconds across all API regions. It is "
             "recommended to rotate tokens at least every 180 days and to store them in a secrets "
             "manager rather than in application source code."},
    {"doc_id": "P11", "category": "API", "product_area": "Rate Limits", "updated": "2025-10-11",
     "title": "API Rate Limiting", 
     "text": "The API enforces a default rate limit of 600 requests per minute per token on the "
             "Business tier and 2,000 requests per minute on the Enterprise tier. Requests exceeding "
             "the limit receive an HTTP 429 response with a Retry-After header indicating the "
             "number of seconds to wait. Sustained high-volume workloads should implement "
             "exponential backoff and, where possible, batch operations using the bulk endpoints "
             "to reduce request counts."},
    {"doc_id": "P12", "category": "API", "product_area": "Webhooks", "updated": "2025-09-08",
     "title": "Configuring Webhooks for File Events",
     "text": "Webhooks notify external systems of file events such as upload, delete, share, and "
             "rename. A webhook is registered with a target URL, a list of subscribed event types, "
             "and a signing secret used to verify payload authenticity via an HMAC-SHA256 signature "
             "in the X-Nimbus-Signature header. Failed webhook deliveries are retried with "
             "exponential backoff for up to 24 hours before the webhook is automatically disabled "
             "and the admin is notified."},
    {"doc_id": "P13", "category": "API", "product_area": "SDKs", "updated": "2025-08-19",
     "title": "Official SDKs and Language Support",
     "text": "NimbusCloud publishes official SDKs for Python, JavaScript/TypeScript, Java, and Go, "
             "all generated from the same OpenAPI specification to keep behavior consistent across "
             "languages. Community-maintained SDKs exist for Ruby and PHP but are not officially "
             "supported. All SDKs implement automatic retry with backoff on 429 and 5xx responses "
             "and expose a lower-level client for advanced use cases not covered by the high-level "
             "convenience methods."},

    {"doc_id": "P14", "category": "Troubleshooting", "product_area": "Sync", "updated": "2025-12-05",
     "title": "Desktop Client Sync Failures",
     "text": "If the desktop sync client shows files stuck in a 'Syncing' state, the most common "
             "causes are a local firewall blocking outbound connections on port 443, a file path "
             "exceeding the 260-character limit on Windows, or a file locked by another application. "
             "Restarting the sync client clears transient locks. Persistent sync failures should be "
             "diagnosed using the sync client's diagnostic log, accessible from the tray icon under "
             "Help, Export Diagnostics."},
    {"doc_id": "P15", "category": "Troubleshooting", "product_area": "Mobile", "updated": "2025-11-09",
     "title": "Mobile App Upload Failures on Cellular Networks",
     "text": "By default, the NimbusCloud mobile app pauses large uploads over 50 megabytes when on "
             "a cellular connection to avoid unexpected data charges, resuming automatically once a "
             "Wi-Fi connection is detected. This behavior can be disabled in the app under Settings, "
             "Data Usage, Allow Cellular Uploads. Uploads that fail even on Wi-Fi are usually caused "
             "by the app being backgrounded by the operating system before the upload completes; "
             "keeping the app in the foreground during large uploads avoids this."},
    {"doc_id": "P16", "category": "Troubleshooting", "product_area": "Sharing", "updated": "2025-07-30",
     "title": "Shared Link Access Denied Errors",
     "text": "An 'Access Denied' error on a shared link most commonly means the link has expired, "
             "the workspace has domain-restricted external sharing enabled, or the recipient's "
             "email domain is on the workspace's blocklist. Admins can review and adjust external "
             "sharing restrictions under Security Settings, External Sharing. Links shared with "
             "'Anyone with the link' permission are never subject to domain restrictions, only "
             "links shared with specific named recipients."},
    {"doc_id": "P17", "category": "Troubleshooting", "product_area": "Performance", "updated": "2025-10-27",
     "title": "Slow Upload and Download Speeds",
     "text": "Upload and download throughput depends on the nearest regional data center, the "
             "number of concurrent transfers, and local network conditions. NimbusCloud automatically "
             "routes traffic to the nearest of twelve global regions. Users experiencing slow "
             "transfers should check whether a VPN is forcing traffic through a distant region, "
             "and should reduce concurrent transfers below the client's default cap of six, since "
             "some corporate networks throttle multiple simultaneous connections to the same host."},

    {"doc_id": "P18", "category": "Onboarding", "product_area": "Provisioning", "updated": "2025-11-14",
     "title": "Bulk User Provisioning for New Workspaces",
     "text": "New Enterprise workspaces can provision users in bulk via CSV upload in the Admin "
             "Console or via SCIM provisioning connected to an identity provider such as Okta or "
             "Azure AD. CSV upload supports up to 5,000 rows per file and requires columns for "
             "email, display name, and role. SCIM provisioning automatically creates, updates, "
             "and deactivates accounts as changes occur in the identity provider, and is the "
             "recommended method for organizations with frequent staff turnover."},
    {"doc_id": "P19", "category": "Onboarding", "product_area": "Migration", "updated": "2025-09-21",
     "title": "Migrating Data from Legacy Storage Providers",
     "text": "The Migration Assistant supports direct transfer from several legacy providers, "
             "preserving folder structure, sharing permissions where mappable, and file timestamps. "
             "Migrations above 5 terabytes should be scheduled with the migrations team to run "
             "during off-peak hours and to enable a dedicated throughput allocation. A dry-run mode "
             "produces a report of files that cannot be mapped automatically, such as permissions "
             "referencing groups that do not exist in NimbusCloud, before the live migration begins."},
    {"doc_id": "P20", "category": "Onboarding", "product_area": "Training", "updated": "2025-08-12",
     "title": "Admin Onboarding Checklist",
     "text": "New workspace admins should complete four steps in the first week: configure SSO or "
             "confirm password policy, set the default storage overage cap, review external sharing "
             "defaults, and invite at least one additional admin for redundancy. NimbusCloud's "
             "customer success team offers a complimentary 45-minute onboarding call for Business "
             "and Enterprise customers, schedulable from the Help menu."},

    {"doc_id": "P21", "category": "Governance", "product_area": "Retention", "updated": "2025-12-10",
     "title": "Data Retention and Legal Hold Policies",
     "text": "Deleted files move to a Trash folder for 30 days before permanent deletion, after "
             "which recovery is not possible except from backups within the standard 90-day backup "
             "window. Legal hold, available on Enterprise tier, suspends permanent deletion for "
             "files matching a defined scope, such as a user's entire workspace or a specific folder, "
             "until the hold is explicitly released by a compliance admin, overriding the normal "
             "30-day trash expiration."},
    {"doc_id": "P22", "category": "Governance", "product_area": "Audit", "updated": "2025-10-16",
     "title": "Audit Log Retention and Export",
     "text": "Audit logs capture login events, permission changes, sharing events, and admin "
             "configuration changes. Logs are retained for 180 days on Business tier and 3 years on "
             "Enterprise tier, and can be exported as CSV or streamed continuously to an external "
             "SIEM via the Audit Log Streaming API. Streaming exports are delivered with at-least-once "
             "delivery semantics, so downstream consumers should de-duplicate on the included event ID."},
    {"doc_id": "P23", "category": "Governance", "product_area": "Data Residency", "updated": "2025-09-02",
     "title": "Data Residency Options",
     "text": "Enterprise customers can pin a workspace's primary data storage to a specific region "
             "— currently United States, European Union, or Singapore — to satisfy data residency "
             "requirements. Backups for a region-pinned workspace remain within the same broad "
             "jurisdiction, for example EU backups stay within the European Economic Area. Changing "
             "a workspace's pinned region after creation requires a manual migration project with "
             "the enterprise solutions team rather than a self-service setting."},
    {"doc_id": "P24", "category": "Governance", "product_area": "Retention", "updated": "2025-11-30",
     "title": "Configuring Automated Retention Rules",
     "text": "Retention rules automatically move files matching defined criteria, such as file age "
             "or folder path, to a compliant archive tier or trigger permanent deletion after a set "
             "number of years. Rules are configured in the Compliance Center and require secondary "
             "approval from a second admin before activation, to prevent accidental mass deletion. "
             "Retention rules always defer to an active legal hold, meaning held files are never "
             "deleted even if a retention rule would otherwise remove them."},
]
print(f"Loaded {len(PARENT_DOCS)} parent documents across categories:",
      sorted(set(d['category'] for d in PARENT_DOCS)))


In [ ]:
def split_into_chunks(text: str, max_sentences: int = 2) -> List[str]:
    '''Simple sentence-based chunker used to create child chunks from each parent document.'''
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i + max_sentences]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks


CHILD_CHUNKS = []  # each: {chunk_id, parent_id, text, category, product_area, updated, title}
for parent in PARENT_DOCS:
    pieces = split_into_chunks(parent["text"], max_sentences=2)
    for i, piece in enumerate(pieces):
        CHILD_CHUNKS.append({
            "chunk_id": f"{parent['doc_id']}-C{i+1}",
            "parent_id": parent["doc_id"],
            "text": piece,
            "category": parent["category"],
            "product_area": parent["product_area"],
            "updated": parent["updated"],
            "title": parent["title"],
        })

PARENT_BY_ID = {p["doc_id"]: p for p in PARENT_DOCS}
print(f"Generated {len(CHILD_CHUNKS)} child chunks from {len(PARENT_DOCS)} parent documents.")
pd.DataFrame(CHILD_CHUNKS)[["chunk_id", "parent_id", "category", "text"]].head(6)


## Build the retrieval indexes

Two indexes are built once and reused throughout the notebook:

1. A **dense index**: embeddings of every child chunk using `text-embedding-3-small`.
2. A **sparse index**: a BM25 index over the same child chunks using `rank_bm25`.

Both operate on identical chunk boundaries so that later comparisons between dense, sparse,
and hybrid retrieval are apples-to-apples.


In [ ]:
from rank_bm25 import BM25Okapi

CHUNK_TEXTS = [c["text"] for c in CHILD_CHUNKS]

# ---- Dense index ----
print("Embedding", len(CHUNK_TEXTS), "chunks with", EMBED_MODEL, "...")
DENSE_INDEX = embed_texts(CHUNK_TEXTS)
print("Dense index shape:", DENSE_INDEX.shape)

# ---- Sparse (BM25) index ----
def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

TOKENIZED_CORPUS = [tokenize(t) for t in CHUNK_TEXTS]
BM25_INDEX = BM25Okapi(TOKENIZED_CORPUS)
print("BM25 index built over", len(TOKENIZED_CORPUS), "chunks.")


In [ ]:
def dense_search(query: str, k: int = 5, allowed_idx: Optional[List[int]] = None) -> List[Tuple[int, float]]:
    '''Dense (embedding) retrieval. Returns list of (chunk_index, score) sorted descending.'''
    q_vec = embed_texts([query])[0]
    if allowed_idx is None:
        scores = DENSE_INDEX @ q_vec
        order = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in order]
    else:
        sub_scores = DENSE_INDEX[allowed_idx] @ q_vec
        order = np.argsort(-sub_scores)[:k]
        return [(allowed_idx[i], float(sub_scores[i])) for i in order]


def sparse_search(query: str, k: int = 5, allowed_idx: Optional[List[int]] = None) -> List[Tuple[int, float]]:
    '''Sparse (BM25) retrieval. Returns list of (chunk_index, score) sorted descending.'''
    scores = BM25_INDEX.get_scores(tokenize(query))
    if allowed_idx is not None:
        mask = np.full(len(scores), -np.inf)
        mask[allowed_idx] = scores[allowed_idx]
        scores = mask
    order = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in order]


def show_results(results: List[Tuple[int, float]], title: str = ""):
    rows = []
    for idx, score in results:
        c = CHILD_CHUNKS[idx]
        rows.append({"score": round(score, 4), "chunk_id": c["chunk_id"],
                      "category": c["category"], "text": c["text"][:100] + "..."})
    df = pd.DataFrame(rows)
    if title:
        print(title)
    return df


---
# Section 4 — Context Improvements

Once the right chunks are retrieved, how they are packaged for the generator still matters:
irrelevant sentences waste context budget, position in the prompt affects how well the LLM
uses each chunk, and a chunk's small size for precise retrieval may be too small for
sufficient answer context. This section addresses all three.


## 4.1 Contextual Compression

Retrieved chunks often contain sentences that are not actually relevant to the query, even if
the chunk as a whole scored well. Contextual compression asks an LLM to extract only the
sentences from each retrieved chunk that are relevant to the specific query, shrinking the
context passed to the final generation step and reducing distraction from irrelevant text.


In [ ]:
def compress_chunk(query: str, chunk_text: str) -> str:
    system = ("Extract only the sentence(s) from the passage below that are directly relevant "
              "to answering the query. Return the extracted text verbatim with no commentary. "
              "If nothing is relevant, return an empty string.")
    prompt = f"Query: {query}\n\nPassage: {chunk_text}"
    return chat(prompt, system=system, temperature=0.0, max_tokens=150)


compression_query = "what happens if my credit card payment fails"
raw_hits = dense_search(compression_query, k=4)
print("Before compression:")
for idx, _ in raw_hits:
    print(" -", CHILD_CHUNKS[idx]["text"])

print("\nAfter contextual compression:")
compressed_chunks = []
for idx, _ in raw_hits:
    compressed = compress_chunk(compression_query, CHILD_CHUNKS[idx]["text"])
    compressed_chunks.append(compressed)
    print(" -", compressed if compressed else "[filtered out - not relevant]")


## 4.2 Chunk Reordering

LLMs are known to attend less reliably to information placed in the middle of a long context
window than to information at the very start or very end (the "lost in the middle" effect).
Chunk reordering keeps the ranking from retrieval but re-arranges the physical order passed
to the prompt so that the most relevant chunks sit at the beginning and end, with less
relevant chunks pushed to the middle.


In [ ]:
def reorder_for_lost_in_middle(ranked_idx: List[int]) -> List[int]:
    '''Given chunk indices already sorted best-first, interleave them so the strongest
    chunks land at the start and end of the sequence rather than all at the front.'''
    result = [None] * len(ranked_idx)
    left, right = 0, len(ranked_idx) - 1
    for i, idx in enumerate(ranked_idx):
        if i % 2 == 0:
            result[left] = idx
            left += 1
        else:
            result[right] = idx
            right -= 1
    return result


ranked_for_reorder = [idx for idx, _ in dense_search("storage overage billing rules", k=6)]
print("Original relevance order (best first): ", [CHILD_CHUNKS[i]['chunk_id'] for i in ranked_for_reorder])
reordered = reorder_for_lost_in_middle(ranked_for_reorder)
print("Reordered for prompt (best at edges):  ", [CHILD_CHUNKS[i]['chunk_id'] for i in reordered])


## 4.3 Parent-Child Retrieval

Small chunks retrieve precisely (a 1-2 sentence chunk closely matches a narrow query) but may
lack surrounding context needed for a complete answer. Parent-child retrieval resolves this
by retrieving on the small child chunks, then substituting in each child's full parent
document as the context actually sent to the generator.


In [ ]:
def parent_child_retrieve(query: str, k: int = 3) -> List[Dict]:
    child_hits = dense_search(query, k=k)
    seen_parents = set()
    results = []
    for idx, score in child_hits:
        child = CHILD_CHUNKS[idx]
        parent_id = child["parent_id"]
        if parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)
        parent = PARENT_BY_ID[parent_id]
        results.append({
            "matched_child": child["text"],
            "score": score,
            "parent_id": parent_id,
            "parent_title": parent["title"],
            "parent_text": parent["text"],
        })
    return results


pc_query = "what's the retry behavior for failed webhook deliveries"
pc_results = parent_child_retrieve(pc_query, k=3)
for r in pc_results:
    print(f"Matched child chunk (score {r['score']:.3f}): {r['matched_child']}")
    print(f"-> Returning full parent document '{r['parent_title']}' for context:")
    print("   " + r["parent_text"])
    print()


## 4.4 Metadata Filtering

Retrieval quality improves substantially when structured metadata (category, product area,
recency) is used to narrow the search space before or after semantic ranking, especially when
the question implies a filter ("recently updated", "billing-related", a named category) that
pure text similarity cannot reliably infer. Here an LLM extracts a structured filter from the
natural-language query, which is then applied against the corpus metadata.


In [ ]:
VALID_CATEGORIES = sorted(set(c["category"] for c in CHILD_CHUNKS))

def extract_metadata_filter(query: str) -> Dict:
    system = (f"Extract a metadata filter from the user's query for a support knowledge base. "
              f"Valid categories: {VALID_CATEGORIES}. Return a JSON object with optional keys "
              f"'category' (one of the valid categories or null) and 'min_updated' (a date "
              f"string 'YYYY-MM-DD' if the user implies recency, else null). Return JSON only.")
    raw = chat(query, system=system, temperature=0.0)
    try:
        return json.loads(re.search(r"\{.*\}", raw, re.S).group())
    except Exception:
        return {"category": None, "min_updated": None}


def filtered_retrieve(query: str, k: int = 5) -> Tuple[List[Tuple[int, float]], Dict]:
    filt = extract_metadata_filter(query)
    allowed = list(range(len(CHILD_CHUNKS)))
    if filt.get("category"):
        allowed = [i for i in allowed if CHILD_CHUNKS[i]["category"] == filt["category"]]
    if filt.get("min_updated"):
        allowed = [i for i in allowed if CHILD_CHUNKS[i]["updated"] >= filt["min_updated"]]
    hits = dense_search(query, k=k, allowed_idx=allowed if allowed else None)
    return hits, filt


meta_query = "show me the most recently updated security policy about encryption"
meta_hits, extracted_filter = filtered_retrieve(meta_query)
print("Query:", meta_query)
print("Extracted metadata filter:", extracted_filter)
print("\nFiltered retrieval results:")
display(show_results(meta_hits))
